# TTM — TinyStories Small-4M đối chứng công bằng · 1.024/2.048 bước

**Operation**: `TTM_PM_TINYSTORIES_MATCHED_4M_1024_2048_COLAB_RUNNER_SOURCE_REPAIR_AND_INDEPENDENT_QA_ONEPACK`.

Đây là notebook **thay thế** phiên bản 40 bước, tokenizer raw-byte vocab258, 2 layer/d128. Cấu hình đúng P02 Small-4M: **3.749.376 tham số, 4 layers, d256, 4 heads, context256, BPE2048 TRAIN-only, 2.048 TinyStories TRAIN / 256 VAL, 4.096 valid next-token targets/optimizer step**.

Mốc A: new random GPT@1024 đối chứng old T4@1024; Mốc B: control@2048 và replica@2048 cùng ngân sách/cùng cosine schedule2048, khác seed đã cố định. Cặp 2048 chỉ được tuyên bố cải thiện nếu có can thiệp riêng đã prereg; với cùng thuật toán, hai kết quả chỉ đo độ biến thiên giữa seeds.

**Mặc định:** `MODE="SOURCE_QA"`, không Torch/GPU/private data/TEST. `NATIVE_QA` thử khôi phục checkpoint trên mô hình toy CPU tổng hợp; `CAPACITY` là benchmark 4M thật trên tensor ngẫu nhiên, không phải science; `PREFLIGHT` xác minh hai byte private TRAIN/VAL exact khi chủ sở hữu chọn. `TRAIN`, `RESUME`, `COMPARE` có chốt owner Q4 exact. Không tự đọc Drive, không tự tìm dataset, không huấn luyện khi Run All mặc định, không tự viết GO. Python3.12 và Tesla T4 bắt buộc cho CAPACITY/TRAIN.

In [ ]:
import hashlib, json, pathlib, platform, subprocess, sys, urllib.request

MODE = "SOURCE_QA"  # SOURCE_QA | NATIVE_QA | CAPACITY | PREFLIGHT | TRAIN | RESUME | COMPARE
ARM = "replica_1024"  # replica_1024 | control_2048 | replica_2048
RUN_ID = "UNSET_NO_Q4_GO"
RESUME_FROM_STEP = None  # only for RESUME: 256,512,768,1024,1536
WORKSPACE = pathlib.Path("/content/TTM_MATCHED_4M_ONEPACK")
PACK_PATH = "/content/TTM_PRIVATE_INPUT/T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"
PACK_MANIFEST_PATH = "/content/TTM_PRIVATE_INPUT/T4_PRIVATE_PACK_MANIFEST.json"
CAPACITY_RECEIPT = "/content/TTM_MATCHED_4M_CAPACITY.json"
OWNER_GO_JSON = "/content/TTM_MATCHED_4M_OWNER_Q4_GO.json"
RUN_ROOT = "/content/TTM_MATCHED_4M_RUNS"
LEFT_CHECKPOINT = "/content/OWNER_CHECKPOINTS/LEFT/checkpoint.pt"
RIGHT_CHECKPOINT = "/content/OWNER_CHECKPOINTS/RIGHT/checkpoint.pt"
COMPARE_RESULT = "/content/TTM_MATCHED_4M_COMPARISON.json"

assert platform.python_version_tuple()[:2] == ("3","12"), "PYTHON312_REQUIRED"
assert MODE in {"SOURCE_QA","NATIVE_QA","CAPACITY","PREFLIGHT","TRAIN","RESUME","COMPARE"}
RUNNER_COMMIT = "e061c8b6559dc793947524aec8ef01fa47bb54a4"
RUNNER_GIT_BLOB = "52d273e8da416832b4ecba76ed0c618545ceaf98"
SOURCE_COMMIT = "59ee3f30b996405fddc087ab60102b59a021f8ea"
PINNED_SOURCES = {
 "src/p02/transformer_core.py":"9762343e06cf800b50574c2871ae1ed58537fb18",
 "tools/pilot_t4/integrated/t4_capacity_smoke.py":"c9a6f256ba975228683ee7de2a1090e649110e97",
 "tools/pilot_t4/integrated/t4_eval.py":"bef006bdff173fba6d8ba4aa0ef687a1732fd7c4",
 "tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py":"76c4d0155cd0736919f41682f240b47ad3d7c428"
}
print("MODE", MODE, "ARM", ARM, "PYTHON", platform.python_version())


In [ ]:
def exact_git_blob(data: bytes):
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\x00" + data).hexdigest()

def fetch_pinned(commit, path, expected, destination):
    url = "https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/" + commit + "/" + path
    with urllib.request.urlopen(url, timeout=35) as response:
        raw = response.read(400_000)
    assert exact_git_blob(raw) == expected, "EXACT_SOURCE_BLOB_MISMATCH:"+path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == raw, "LOCAL_SOURCE_ALREADY_DIFFERS:"+str(destination)
    else:
        with destination.open("xb") as f:
            f.write(raw)
    return hashlib.sha256(raw).hexdigest()

WORKSPACE.mkdir(parents=True, exist_ok=True)
RUNNER_PATH = WORKSPACE / "src/ttm_new_gpt_baseline/runner.py"
verified = {"runner": fetch_pinned(RUNNER_COMMIT, "src/ttm_new_gpt_baseline/runner.py",
                                   RUNNER_GIT_BLOB, RUNNER_PATH)}
for relative, sha1 in PINNED_SOURCES.items():
    verified[relative] = fetch_pinned(SOURCE_COMMIT, relative, sha1, WORKSPACE/relative)
print("EXACT_SOURCE_PINS_PASS", len(verified))
print(json.dumps(verified, indent=2, sort_keys=True))


In [ ]:
command = [sys.executable, str(RUNNER_PATH)]
if MODE == "SOURCE_QA":
    command += ["qa"]
    timeout_seconds = 60
elif MODE == "NATIVE_QA":
    command += ["native-qa", "--sources", str(WORKSPACE),
                "--out", str(WORKSPACE/"TTM_NATIVE_QA_RECEIPT.json")]
    timeout_seconds = 180
elif MODE == "CAPACITY":
    command += ["capacity", "--sources", str(WORKSPACE), "--out", CAPACITY_RECEIPT]
    timeout_seconds = 210
elif MODE == "PREFLIGHT":
    command += ["preflight", "--sources", str(WORKSPACE),
                "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH]
    timeout_seconds = 120
else:
    assert MODE in {"TRAIN","RESUME","COMPARE"}
    assert RUN_ID != "UNSET_NO_Q4_GO", "OWNER_RUN_ID_AND_EXACT_Q4_REQUIRED"
    assert pathlib.Path(OWNER_GO_JSON).is_file(), "EXPLICIT_Q4_GO_FILE_MISSING"
    assert ARM in {"replica_1024","control_2048","replica_2048"}
    if MODE in ("TRAIN","RESUME"):
        command += ["train", "--sources", str(WORKSPACE),
            "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH,
            "--arm", ARM, "--run-id", RUN_ID, "--go", OWNER_GO_JSON,
            "--capacity", CAPACITY_RECEIPT, "--out", RUN_ROOT+"/"+ARM+"__"+RUN_ID]
        if MODE == "RESUME":
            assert RESUME_FROM_STEP in (256,512,768,1024,1536), "EXPLICIT_RESUME_CHECKPOINT_STEP_REQUIRED"
            command += ["--resume-step", str(RESUME_FROM_STEP)]
        timeout_seconds = 5900
    else:
        command += ["compare", "--sources", str(WORKSPACE),
            "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH,
            "--arm", ARM, "--run-id", RUN_ID, "--go", OWNER_GO_JSON,
            "--left", LEFT_CHECKPOINT, "--right", RIGHT_CHECKPOINT,
            "--out", COMPARE_RESULT]
        timeout_seconds = 1200
print("ACTION", MODE, "SCIENCE", MODE in ("TRAIN","RESUME","COMPARE"))
# NO shell=True, no git writes, no automatic Drive mounting or historical TEST read.
result = subprocess.run(command, capture_output=True, text=True,
                        timeout=timeout_seconds, check=False)
print("EXIT_CODE", result.returncode)
print("STDOUT", result.stdout[-18000:])
print("STDERR", result.stderr[-18000:])
assert result.returncode == 0, "FAIL_CLOSED_ACTION_NOT_SUCCESSFUL"


## Cổng khoa học (Q4) và chứng cứ

- `SOURCE_QA`: kiểm thử source thuần (Python stdlib), không chứng minh native 4M/T4.
- `NATIVE_QA`: phép kiểm nhỏ 4 bước, split 2+2 trên CPU synthetic, so tham số/loss & thử phá SHA; **không phải** resume parity của 4M FP16 trên T4.
- `CAPACITY`: T4 real compute trên **random tensors** 2 warmup + 3 timed, full P02 3.749.376 params, context256, 4.096 labels/step; chi phí hai nhánh 2.048 steps dự báo từ worst latency. Đây là estimate, không bao gồm phân bố độ dài cửa sổ thực, IO/VAL/checkpoint; không đủ Q4.
- `PREFLIGHT`: owner chủ động cung cấp exactly historical TRAIN/VAL NPZ+manifest. SHA256 pack `184f68b4...`, manifest `ef2e6e67...`; dữ liệu nhạy cảm KHÔNG upload lên GitHub.
- `TRAIN` / `RESUME`: yêu cầu tệp Q4 quyết định riêng trùng exact runner blob, recipe, pack, capacity, arm, run ID, rights reference. Không tự tạo GO. Checkpoint READY-last tại 256/512/768/1024/(1536/2048); resume cùng one-shot RUN_LOCK, không overwrite và không nhảy checkpoint. Cần storage bền vững do owner cung cấp nếu Colab bị reset; đường dẫn `/content` mặc định sẽ mất khi phiên bị xóa.
- `COMPARE`: yêu cầu Q4 action COMPARE, hai checkpoint SHA được khóa trong GO và cùng bước 1.024 hoặc 2.048. Dùng 256 VAL truyện lịch sử đã có, paired bootstrap 2.000 lần. Đây là developmental evaluation; **không mở lại TEST 64 truyện đã tiêu thụ**.

**Khác biệt cần giữ:** A/B/C/D checkpoint EmbeddingGemma2 2.048 trước đây KHÔNG tự động làm đối chứng cho một kiến trúc/schedule khác. Một cặp 2.048 phải được chứng minh cùng recipe và data bytes; nếu không đủ chứng cứ => HOLD so sánh 2.048.

**Kết quả đầu ra của notebook chưa được chạy ở lượt triển khai GitHub:** `OWNER_Q4=HOLD · T4_MODEL_TRAIN=NOT_EXECUTED · PRIVATE_PACK_NOT_READ · TEST=NOT_ACCESSED`.